# 02 - Training Model ANN
**Proyek WASPADA** — Sistem Deteksi Fraud Transaksi Keuangan

Tahapan:
1. Load hasil preprocessing
2. Bangun arsitektur ANN
3. Training model
4. Evaluasi performa
5. Simpan model

## 1. Import Library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, f1_score
)
import seaborn as sns

print('Semua library berhasil diimport!')

## 2. Load Hasil Preprocessing

In [ ]:
X_train = np.load('processed/X_train.npy')
y_train = np.load('processed/y_train.npy')
X_test  = np.load('processed/X_test.npy')
y_test  = np.load('processed/y_test.npy')

print(f'X_train : {X_train.shape}')
print(f'y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_test  : {y_test.shape}')
print(f'\nJumlah fitur input : {X_train.shape[1]}')

## 3. Bangun Arsitektur ANN

Arsitektur yang digunakan:
- **Input layer** : 30 fitur
- **Hidden layer 1** : 64 neuron, aktivasi ReLU, Dropout 30%
- **Hidden layer 2** : 32 neuron, aktivasi ReLU, Dropout 20%
- **Output layer** : 1 neuron, aktivasi Sigmoid (output probabilitas 0–1)

In [ ]:
input_dim = X_train.shape[1]

model = keras.Sequential([
    layers.Input(shape=(input_dim,)),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 4. Training Model

In [ ]:
os.makedirs('../backend/model', exist_ok=True)

# Early stopping: berhenti jika val_loss tidak membaik selama 5 epoch
early_stop = EarlyStopping(
    monitor='val_loss', patience=5,
    restore_best_weights=True, verbose=1
)

# Simpan model terbaik secara otomatis
checkpoint = ModelCheckpoint(
    '../backend/model/ann_model.h5',
    monitor='val_loss', save_best_only=True, verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

print('\nTraining selesai!')

## 5. Visualisasi Hasil Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Grafik Loss
axes[0].plot(history.history['loss'],     label='Train Loss', color='#1D9E75')
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='#D85A30')
axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Grafik Accuracy
axes[1].plot(history.history['accuracy'],     label='Train Accuracy', color='#1D9E75')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy',   color='#D85A30')
axes[1].set_title('Training & Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('ann_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan: ann_training_history.png')

## 6. Evaluasi Model pada Data Test

In [ ]:
# Prediksi probabilitas
y_prob = model.predict(X_test).flatten()

# Threshold 0.5 untuk konversi ke label biner
y_pred = (y_prob >= 0.5).astype(int)

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

auc = roc_auc_score(y_test, y_prob)
f1  = f1_score(y_test, y_pred)
print(f'AUC-ROC Score : {auc:.4f}')
print(f'F1-Score      : {f1:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0],
            cmap='Greens', xticklabels=['Normal','Fraud'],
            yticklabels=['Normal','Fraud'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Aktual')
axes[0].set_xlabel('Prediksi')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#1D9E75', lw=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0,1],[0,1], color='gray', linestyle='--')
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('ann_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan: ann_evaluation.png')

## 7. Simpan Probabilitas untuk Fuzzy Sugeno

In [ ]:
# Simpan probabilitas prediksi test set
# Ini akan dipakai sebagai input saat merancang membership function Fuzzy
np.save('processed/y_prob_test.npy', y_prob)

print('Distribusi probabilitas prediksi:')
print(f'  Min  : {y_prob.min():.4f}')
print(f'  Max  : {y_prob.max():.4f}')
print(f'  Mean : {y_prob.mean():.4f}')
print(f'  Std  : {y_prob.std():.4f}')

plt.figure(figsize=(8, 4))
plt.hist(y_prob[y_test==0], bins=50, alpha=0.6, color='#1D9E75', label='Normal')
plt.hist(y_prob[y_test==1], bins=50, alpha=0.6, color='#D85A30', label='Fraud')
plt.axvline(x=0.4, color='gray', linestyle='--', label='Batas Aman (0.4)')
plt.axvline(x=0.6, color='black', linestyle='--', label='Batas Sangat Mencurigakan (0.6)')
plt.title('Distribusi Probabilitas ANN')
plt.xlabel('Probabilitas Fraud')
plt.ylabel('Jumlah')
plt.legend()
plt.tight_layout()
plt.savefig('ann_prob_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan: ann_prob_distribution.png')

## Ringkasan Training ANN

| Komponen | Nilai |
|---|---|
| Input layer | 30 fitur |
| Hidden layer 1 | 64 neuron, ReLU, Dropout 30% |
| Hidden layer 2 | 32 neuron, ReLU, Dropout 20% |
| Output layer | 1 neuron, Sigmoid |
| Optimizer | Adam (lr=0.001) |
| Loss function | Binary Crossentropy |
| Model tersimpan | backend/model/ann_model.h5 |